# Backpropagation Algorithm for LLMs

This notebook provides a hands-on introduction to the backpropagation algorithm, which is the core training mechanism for all neural networks including Large Language Models.

## Overview

- **Part 1**: Chain Rule & Computational Graph
- **Part 2**: Manual Backpropagation on a Simple Network
- **Part 3**: PyTorch Autograd — Automatic Differentiation
- **Part 4**: Backpropagation in a Multi-Layer Network
- **Part 5**: Backpropagation in a Transformer-like Architecture
- **Part 6**: Numerical Gradient Checking
- **Part 7**: Common Pitfalls & Debugging Tips

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np

print("PyTorch version:", torch.__version__)
torch.manual_seed(42)

---

## Part 1: Chain Rule & Computational Graph

Backpropagation is essentially the **chain rule of calculus** applied systematically through a computational graph.

### The Chain Rule

If $y = f(g(x))$, then:

$$\frac{dy}{dx} = \frac{dy}{dg} \cdot \frac{dg}{dx}$$

For a multi-layer network with loss $L$:

$$\frac{\partial L}{\partial w} = \frac{\partial L}{\partial a_3} \cdot \frac{\partial a_3}{\partial z_3} \cdot \frac{\partial z_3}{\partial a_2} \cdot \frac{\partial a_2}{\partial z_2} \cdot \frac{\partial z_2}{\partial w}$$

where $z_i$ is the pre-activation and $a_i$ is the post-activation at layer $i$.

In [ ]:
# Simple example: f(x) = sin(x^2)
# df/dx = cos(x^2) * 2x  (chain rule!)

x = torch.tensor(3.0, requires_grad=True)

# Forward pass
y = x ** 2
z = torch.sin(y)

# Backward pass
z.backward()

print(f"x = {x.item()}")
print(f"y = x² = {y.item()}")
print(f"z = sin(y) = {z.item():.6f}")
print(f"dz/dx (autograd) = {x.grad.item():.6f}")
print(f"dz/dx (analytical) = {math.cos(x.item()**2) * 2 * x.item():.6f}")

### Computational Graph Visualization

PyTorch builds a **directed acyclic graph (DAG)** during the forward pass:

```
  x ──→ [pow] ──→ y ──→ [sin] ──→ z (loss)
              ²              sin
```

During backward pass, gradients flow in reverse:

```
  x ←── [pow] ←── y ←── [sin] ←── z (loss)
         ∂y/∂x=2x      ∂z/∂y=cos(y)
```

In [ ]:
# Inspect the computational graph
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2
z = torch.sin(y)

print(f"z.grad_fn = {z.grad_fn}")
print(f"z.grad_fn.next_functions = {z.grad_fn.next_functions}")
print(f"y.grad_fn = {y.grad_fn}")
print(f"y.grad_fn.next_functions = {y.grad_fn.next_functions}")

---

## Part 2: Manual Backpropagation on a Simple Network

Let's implement backpropagation **from scratch** for a 2-layer network:

$$\hat{y} = \sigma(w_2 \cdot \text{ReLU}(w_1 \cdot x + b_1) + b_2)$$

with MSE loss: $L = \frac{1}{2}(\hat{y} - y)^2$

In [ ]:
torch.manual_seed(42)

# Network parameters
w1 = torch.randn(3, 2, dtype=torch.float64)  # input_dim=2, hidden_dim=3
b1 = torch.randn(3, dtype=torch.float64)
w2 = torch.randn(1, 3, dtype=torch.float64)  # hidden_dim=3, output_dim=1
b2 = torch.randn(1, dtype=torch.float64)

# Input and target
x = torch.randn(2, dtype=torch.float64)
y_true = torch.tensor([1.0], dtype=torch.float64)

print(f"Input x: {x.tolist()}")
print(f"Target y: {y_true.item()}")
print(f"w1 shape: {w1.shape}, w2 shape: {w2.shape}")

In [ ]:
# ===== FORWARD PASS =====

# Layer 1: z1 = w1 @ x + b1,  a1 = ReLU(z1)
z1 = w1 @ x + b1
a1 = torch.relu(z1)

# Layer 2: z2 = w2 @ a1 + b2,  a2 = sigmoid(z2)
z2 = w2 @ a1 + b2
a2 = torch.sigmoid(z2)

# Loss: L = 0.5 * (a2 - y_true)^2
loss = 0.5 * (a2 - y_true) ** 2

print(f"z1 (pre-activation): {z1.tolist()}")
print(f"a1 (post-ReLU):      {a1.tolist()}")
print(f"z2 (pre-activation): {z2.item():.6f}")
print(f"a2 (prediction):     {a2.item():.6f}")
print(f"Loss:                {loss.item():.6f}")

In [ ]:
# ===== BACKWARD PASS (Manual!) =====

# Step 1: dL/da2 = (a2 - y_true)
dL_da2 = a2 - y_true
print(f"dL/da2 = {dL_da2.item():.6f}")

# Step 2: da2/dz2 = sigmoid(z2) * (1 - sigmoid(z2))
sigmoid_deriv = a2 * (1 - a2)
dL_dz2 = dL_da2 * sigmoid_deriv
print(f"dL/dz2 = {dL_dz2.item():.6f}")

# Step 3: Gradients for w2 and b2
# z2 = w2 @ a1 + b2  =>  dL/dw2 = dL/dz2 * dz2/dw2 = dL/dz2 * a1^T
dL_dw2 = dL_dz2.unsqueeze(1) @ a1.unsqueeze(0)  # (1,1) @ (1,3) -> (1,3)
dL_db2 = dL_dz2  # (1,)
print(f"dL/dw2 = {dL_dw2.tolist()}")
print(f"dL/db2 = {dL_db2.item():.6f}")

# Step 4: Propagate gradient through w2 to a1
# dL/da1 = dL/dz2 * dz2/da1 = w2^T @ dL/dz2
dL_da1 = w2.T @ dL_dz2  # (3,1) @ (1,) -> (3,)
print(f"dL/da1 = {dL_da1.tolist()}")

# Step 5: Through ReLU: dL/dz1 = dL/da1 * (z1 > 0)
relu_mask = (z1 > 0).float()
dL_dz1 = dL_da1 * relu_mask
print(f"dL/dz1 = {dL_dz1.tolist()}")

# Step 6: Gradients for w1 and b1
# z1 = w1 @ x + b1  =>  dL/dw1 = dL/dz1 * x^T
dL_dw1 = dL_dz1.unsqueeze(1) @ x.unsqueeze(0)  # (3,1) @ (1,2) -> (3,2)
dL_db1 = dL_dz1  # (3,)
print(f"dL/dw1 = {dL_dw1.tolist()}")
print(f"dL/db1 = {dL_db1.tolist()}")

In [ ]:
# ===== VERIFY with PyTorch Autograd =====

w1_auto = w1.clone().requires_grad_(True)
b1_auto = b1.clone().requires_grad_(True)
w2_auto = w2.clone().requires_grad_(True)
b2_auto = b2.clone().requires_grad_(True)

z1_auto = w1_auto @ x + b1_auto
a1_auto = torch.relu(z1_auto)
z2_auto = w2_auto @ a1_auto + b2_auto
a2_auto = torch.sigmoid(z2_auto)
loss_auto = 0.5 * (a2_auto - y_true) ** 2

loss_auto.backward()

print("=== Gradient Comparison: Manual vs Autograd ===")
print(f"dL/dw1 max diff: {(dL_dw1 - w1_auto.grad).abs().max().item():.2e}")
print(f"dL/db1 max diff: {(dL_db1 - b1_auto.grad).abs().max().item():.2e}")
print(f"dL/dw2 max diff: {(dL_dw2 - w2_auto.grad).abs().max().item():.2e}")
print(f"dL/db2 max diff: {(dL_db2 - b2_auto.grad).abs().max().item():.2e}")
print("\n✓ Manual backpropagation matches PyTorch autograd!")

---

## Part 3: PyTorch Autograd — Automatic Differentiation

PyTorch's `autograd` system automatically tracks operations on tensors with `requires_grad=True` and computes gradients via `backward()`.

In [ ]:
# Basic autograd mechanics

# 1. requires_grad controls gradient tracking
a = torch.tensor([2.0, 3.0], requires_grad=True)
b = torch.tensor([4.0, 5.0], requires_grad=False)  # no gradient needed

c = a * b + a ** 2
loss = c.sum()
loss.backward()

print(f"a.grad = {a.grad}")  # dL/da = b + 2a
print(f"b.grad = {b.grad}")  # None (requires_grad=False)
print(f"Expected dL/da = b + 2a = {(b + 2*a).detach()}")

In [ ]:
# 2. Gradient accumulation — gradients ADD UP on successive backward() calls
w = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

for i in range(3):
    loss = (w ** 2).sum()
    loss.backward()
    print(f"After backward #{i+1}: w.grad = {w.grad.tolist()}")
    # NOTE: gradients accumulate! Must zero them manually:
    w.grad.zero_()

print("\n→ Always call optimizer.zero_grad() or w.grad.zero_() before each iteration!")

In [ ]:
# 3. torch.no_grad() — disable gradient tracking (for inference)
w = torch.randn(3, requires_grad=True)

# Training step (gradients tracked)
loss = (w ** 2).sum()
loss.backward()
print(f"Training: w.grad = {w.grad}")

# Inference step (no gradients, saves memory & computation)
with torch.no_grad():
    output = (w ** 2).sum()
    print(f"Inference: output.requires_grad = {output.requires_grad}")

# Alternative: .detach() removes a tensor from the computation graph
detached_w = w.detach()
print(f"detached_w.requires_grad = {detached_w.requires_grad}")

---

## Part 4: Backpropagation in a Multi-Layer Network

Now let's build a complete training loop with backpropagation for a multi-layer network.

In [ ]:
class MultiLayerNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=3):
        super().__init__()
        layers = []
        in_dim = input_dim
        for i in range(num_layers - 1):
            layers.append(nn.Linear(in_dim, hidden_dim))
            layers.append(nn.ReLU())
            in_dim = hidden_dim
        layers.append(nn.Linear(hidden_dim, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

model = MultiLayerNet(input_dim=2, hidden_dim=16, output_dim=1, num_layers=3)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Generate a simple dataset: y = sin(x1) + cos(x2)
torch.manual_seed(42)
n_samples = 200
X = torch.randn(n_samples, 2)
y = (torch.sin(X[:, 0]) + torch.cos(X[:, 1])).unsqueeze(1)

# Split into train/val
n_train = 160
X_train, X_val = X[:n_train], X[n_train:]
y_train, y_val = y[:n_train], y[n_train:]

print(f"Train: {X_train.shape}, Val: {X_val.shape}")

In [ ]:
# ===== Full Training Loop with Backpropagation =====

model = MultiLayerNet(input_dim=2, hidden_dim=16, output_dim=1, num_layers=3)
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

train_losses = []
val_losses = []

for epoch in range(300):
    # --- Forward pass ---
    y_pred = model(X_train)
    loss = criterion(y_pred, y_train)

    # --- Backward pass (backpropagation!) ---
    optimizer.zero_grad()  # Step 1: Clear old gradients
    loss.backward()        # Step 2: Compute gradients via backprop
    optimizer.step()       # Step 3: Update parameters using gradients

    # Validation
    with torch.no_grad():
        val_pred = model(X_val)
        val_loss = criterion(val_pred, y_val)

    train_losses.append(loss.item())
    val_losses.append(val_loss.item())

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1:3d} | Train Loss: {loss.item():.4f} | Val Loss: {val_loss.item():.4f}")

print("\nTraining complete!")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss curves
axes[0].plot(train_losses, label='Train Loss')
axes[0].plot(val_losses, label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].set_yscale('log')

# Prediction vs Ground Truth
with torch.no_grad():
    y_val_pred = model(X_val).squeeze()
axes[1].scatter(y_val.squeeze().numpy(), y_val_pred.numpy(), alpha=0.6)
axes[1].plot([-2, 2], [-2, 2], 'r--', label='Perfect')
axes[1].set_xlabel('Ground Truth')
axes[1].set_ylabel('Prediction')
axes[1].set_title('Val: Prediction vs Ground Truth')
axes[1].legend()

plt.tight_layout()
plt.show()

### Inspecting Gradients During Training

Understanding gradient magnitudes is crucial for diagnosing training issues.

In [ ]:
model = MultiLayerNet(input_dim=2, hidden_dim=16, output_dim=1, num_layers=3)
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

# Track gradient norms per layer across epochs
grad_history = {f'layer_{i}': [] for i in range(3)}
layer_names = ['layer_0', 'layer_1', 'layer_2']

for epoch in range(200):
    y_pred = model(X_train)
    loss = criterion(y_pred, y_train)
    optimizer.zero_grad()
    loss.backward()

    # Record gradient norms for each linear layer
    linear_layers = [m for m in model.net if isinstance(m, nn.Linear)]
    for i, layer in enumerate(linear_layers):
        if layer.weight.grad is not None:
            grad_norm = layer.weight.grad.norm().item()
            grad_history[layer_names[i]].append(grad_norm)

    optimizer.step()

# Plot gradient norms
plt.figure(figsize=(10, 4))
for name in layer_names:
    plt.plot(grad_history[name], label=name)
plt.xlabel('Epoch')
plt.ylabel('Gradient L2 Norm')
plt.title('Gradient Norms per Layer During Training')
plt.legend()
plt.yscale('log')
plt.tight_layout()
plt.show()

---

## Part 5: Backpropagation in a Transformer-like Architecture

Now let's apply backpropagation to a simplified Transformer block — the core building block of LLMs like GPT.

In [ ]:
class SimpleSelfAttention(nn.Module):
    """Single-head self-attention mechanism."""
    def __init__(self, d_model):
        super().__init__()
        self.d_k = d_model
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        # Scaled dot-product attention
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)

        # Causal mask (for autoregressive generation)
        seq_len = x.size(1)
        mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
        scores = scores.masked_fill(mask, float('-inf'))

        attn_weights = torch.softmax(scores, dim=-1)
        attn_output = attn_weights @ V
        return self.W_o(attn_output)


class TransformerBlock(nn.Module):
    """Simplified Transformer block: Attention + FFN with residual connections & layer norm."""
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.attention = SimpleSelfAttention(d_model)
        self.ln1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x):
        # Residual connection + LayerNorm (Pre-LN style, like GPT-2)
        x = x + self.attention(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class MiniGPT(nn.Module):
    """A minimal GPT-like model for demonstration."""
    def __init__(self, vocab_size, d_model, d_ff, n_blocks, max_seq_len):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.blocks = nn.Sequential(*[
            TransformerBlock(d_model, d_ff) for _ in range(n_blocks)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        # Weight tying: share weights between embedding and output head
        self.head.weight = self.token_emb.weight

    def forward(self, input_ids):
        B, T = input_ids.shape
        tok_emb = self.token_emb(input_ids)
        pos = torch.arange(T, device=input_ids.device).unsqueeze(0)
        pos_emb = self.pos_emb(pos)
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.head(x)
        return logits


# Instantiate
torch.manual_seed(42)
vocab_size = 100
d_model = 64
d_ff = 256
n_blocks = 2
max_seq_len = 32

model = MiniGPT(vocab_size, d_model, d_ff, n_blocks, max_seq_len)
total_params = sum(p.numel() for p in model.parameters())
print(f"MiniGPT parameters: {total_params:,}")
print(f"\nModel architecture:")
print(model)

In [ ]:
# ===== Training the MiniGPT with Backpropagation =====

torch.manual_seed(42)
model = MiniGPT(vocab_size, d_model, d_ff, n_blocks, max_seq_len)

# Generate random training data (token IDs)
batch_size = 8
seq_len = 16

X_train_ids = torch.randint(0, vocab_size, (64, seq_len))
y_train_ids = torch.randint(0, vocab_size, (64, seq_len))

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

losses = []

for epoch in range(200):
    # Mini-batch
    idx = torch.randint(0, len(X_train_ids), (batch_size,))
    x_batch = X_train_ids[idx]
    y_batch = y_train_ids[idx]

    # Forward
    logits = model(x_batch)  # (B, T, vocab_size)

    # Reshape for cross-entropy: (B*T, vocab_size) vs (B*T,)
    loss = criterion(logits.view(-1, vocab_size), y_batch.view(-1))

    # Backward (backpropagation!)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())
    if (epoch + 1) % 40 == 0:
        print(f"Epoch {epoch+1:3d} | Loss: {loss.item():.4f}")

plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Cross-Entropy Loss')
plt.title('MiniGPT Training Loss')
plt.yscale('log')
plt.tight_layout()
plt.show()

In [ ]:
# ===== Inspect gradient flow through the Transformer =====

torch.manual_seed(42)
model = MiniGPT(vocab_size, d_model, d_ff, n_blocks, max_seq_len)
criterion = nn.CrossEntropyLoss()

x_sample = torch.randint(0, vocab_size, (4, seq_len))
y_sample = torch.randint(0, vocab_size, (4, seq_len))

logits = model(x_sample)
loss = criterion(logits.view(-1, vocab_size), y_sample.view(-1))
loss.backward()

print("Gradient norms for each parameter group:")
print("=" * 50)
for name, param in model.named_parameters():
    if param.grad is not None:
        grad_norm = param.grad.norm().item()
        grad_mean = param.grad.mean().item()
        grad_std = param.grad.std().item()
        print(f"{name:40s} | norm={grad_norm:.4e} | mean={grad_mean:.4e} | std={grad_std:.4e}")
    else:
        print(f"{name:40s} | NO GRADIENT")

---

## Part 6: Numerical Gradient Checking

A crucial debugging technique: verify analytical gradients against **numerical (finite difference) gradients**.

$$\frac{\partial f}{\partial w_i} \approx \frac{f(w_i + \epsilon) - f(w_i - \epsilon)}{2\epsilon}$$

In [ ]:
def numerical_gradient(model, loss_fn, x, y, param_name, eps=1e-5):
    """Compute numerical gradient using central finite differences."""
    param = dict(model.named_parameters())[param_name]
    grad = torch.zeros_like(param.data)

    for idx in np.ndindex(param.data.shape):
        original = param.data[idx].item()

        param.data[idx] = original + eps
        loss_plus = loss_fn(model(x), y).item()

        param.data[idx] = original - eps
        loss_minus = loss_fn(model(x), y).item()

        grad[idx] = (loss_plus - loss_minus) / (2 * eps)
        param.data[idx] = original

    return grad


# Test on a small model
torch.manual_seed(42)
small_model = MultiLayerNet(input_dim=2, hidden_dim=4, output_dim=1, num_layers=2)
criterion = nn.MSELoss()

x_test = torch.randn(3, 2)
y_test = torch.randn(3, 1)

# Analytical gradient
loss = criterion(small_model(x_test), y_test)
loss.backward()

# Compare for each parameter
print("Numerical Gradient Check")
print("=" * 60)
for name, param in small_model.named_parameters():
    if param.grad is not None:
        num_grad = numerical_gradient(small_model, criterion, x_test, y_test, name)
        diff = (param.grad - num_grad).abs().max().item()
        rel_diff = diff / (param.grad.abs().max().item() + 1e-8)
        status = "✓ PASS" if rel_diff < 1e-4 else "✗ FAIL"
        print(f"{name:25s} | max_diff={diff:.2e} | rel_diff={rel_diff:.2e} | {status}")

---

## Part 7: Common Pitfalls & Debugging Tips

### Vanishing & Exploding Gradients

In [ ]:
# Demonstrate vanishing gradients with a deep network (no residual connections)
torch.manual_seed(42)

class DeepNetNoResidual(nn.Module):
    def __init__(self, depth, hidden_dim=32):
        super().__init__()
        layers = []
        for _ in range(depth):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.Sigmoid())  # Sigmoid is prone to vanishing gradients!
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).sum()  # scalar output

depths = [5, 10, 20, 40]
print("Vanishing Gradient Demonstration (Sigmoid activation, no residual connections)")
print("=" * 70)

for depth in depths:
    model = DeepNetNoResidual(depth)
    x = torch.randn(1, 32)
    loss = model(x)
    loss.backward()

    linear_layers = [m for m in model.net if isinstance(m, nn.Linear)]
    first_grad = linear_layers[0].weight.grad.norm().item()
    last_grad = linear_layers[-1].weight.grad.norm().item()
    ratio = first_grad / (last_grad + 1e-30)
    print(f"Depth={depth:2d} | First layer grad norm: {first_grad:.4e} | "
          f"Last layer grad norm: {last_grad:.4e} | Ratio: {ratio:.4e}")

In [ ]:
# Solution: Residual connections + better activations (GELU/ReLU)
class DeepNetWithResidual(nn.Module):
    def __init__(self, depth, hidden_dim=32):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(hidden_dim, hidden_dim) for _ in range(depth)
        ])
        self.act = nn.GELU()  # GELU is better than Sigmoid

    def forward(self, x):
        for layer in self.layers:
            x = x + self.act(layer(x))  # Residual connection!
        return x.sum()

print("With Residual Connections + GELU")
print("=" * 70)

for depth in depths:
    model = DeepNetWithResidual(depth)
    x = torch.randn(1, 32)
    loss = model(x)
    loss.backward()

    first_grad = model.layers[0].weight.grad.norm().item()
    last_grad = model.layers[-1].weight.grad.norm().item()
    ratio = first_grad / (last_grad + 1e-30)
    print(f"Depth={depth:2d} | First layer grad norm: {first_grad:.4e} | "
          f"Last layer grad norm: {last_grad:.4e} | Ratio: {ratio:.4e}")

print("\n→ Residual connections keep gradients flowing even in very deep networks!")
print("  This is why Transformers use residual connections in every block.")

### Gradient Clipping

A practical technique to prevent exploding gradients during training:

In [ ]:
torch.manual_seed(42)
model = MiniGPT(vocab_size=100, d_model=64, d_ff=256, n_blocks=2, max_seq_len=32)
criterion = nn.CrossEntropyLoss()

x = torch.randint(0, 100, (4, 16))
y = torch.randint(0, 100, (4, 16))

logits = model(x)
loss = criterion(logits.view(-1, 100), y.view(-1))
loss.backward()

# Before clipping
total_norm_before = torch.sqrt(sum(p.grad.norm() ** 2 for p in model.parameters() if p.grad is not None))
print(f"Total gradient norm BEFORE clipping: {total_norm_before.item():.4f}")

# Apply gradient clipping (max_norm=1.0)
max_norm = 1.0
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)

# After clipping
total_norm_after = torch.sqrt(sum(p.grad.norm() ** 2 for p in model.parameters() if p.grad is not None))
print(f"Total gradient norm AFTER clipping:  {total_norm_after.item():.4f}")
print(f"\n→ Gradient clipping prevents dangerously large gradient updates!")

### Learning Rate Sensitivity

The learning rate controls how big a step we take in the gradient direction. Too large → divergence; too small → slow training.

In [ ]:
learning_rates = [1e-4, 1e-3, 1e-2, 1e-1]
all_losses = {}

for lr in learning_rates:
    torch.manual_seed(42)
    model = MultiLayerNet(input_dim=2, hidden_dim=16, output_dim=1, num_layers=3)
    criterion = nn.MSELoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    losses = []
    for epoch in range(200):
        y_pred = model(X_train)
        loss = criterion(y_pred, y_train)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    all_losses[lr] = losses
    print(f"lr={lr:.0e} | Final loss: {losses[-1]:.4f}")

plt.figure(figsize=(10, 5))
for lr, losses in all_losses.items():
    plt.plot(losses, label=f'lr={lr:.0e}')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Effect of Learning Rate on Training')
plt.legend()
plt.yscale('log')
plt.tight_layout()
plt.show()

### Key Takeaways

1. **Backpropagation = Chain Rule + Computational Graph**: Gradients are computed layer by layer in reverse order, multiplying local gradients along the way.

2. **Three steps per training iteration**:
   - `optimizer.zero_grad()` — clear old gradients
   - `loss.backward()` — compute new gradients via backprop
   - `optimizer.step()` — update parameters

3. **Vanishing/Exploding Gradients**: Deep networks can suffer from gradients that shrink or grow exponentially. Solutions:
   - Residual connections (used in Transformers)
   - Better activations (GELU > Sigmoid)
   - Gradient clipping
   - Proper weight initialization

4. **Learning Rate Matters**: Too large → divergence; too small → slow convergence. AdamW with learning rate scheduling is the standard for LLM training.

5. **Numerical Gradient Checking**: Always verify your custom gradient implementations against finite differences.